# Porcentajes con dos decimales

Recalcula con dos decimales porcentajes reportados en la tesis.

## Rutas

Las rutas son relativas a la carpeta `notebooks/`. Los datos no forman parte del repositorio (ver README).

In [ ]:
# Comprobación de rutas
import os
PROY = os.path.abspath('..')
print("carpeta del notebook :", os.getcwd())
print("raiz del proyecto    :", PROY)
for _sub in ("Topic/Dataframes", "Topic/Resultados", "csv_parques"):
    _p = os.path.join(PROY, _sub)
    print(f"  {'OK ' if os.path.isdir(_p) else 'FALTA'}  {_sub}")
print("\nSi algo dice FALTA, el kernel no arranco en la carpeta del notebook,\n"
      "o los datos no estan donde se los espera (ver README).")

### Polaridad y reseñas cortas (Secciones 3.4.2 y 3.4.4)

In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

BD    = ".."
DFDIR = f"{BD}/Topic/Dataframes"

# ---------- Lecturas (solo lectura, nada se escribe) ----------
base   = pd.read_pickle(f"{BD}/df_generic_cleaned.pkl")          # mismo que EDA y M4
prep   = pickle.load(open(f"{DFDIR}/prep_lemmatizado.pkl", "rb"))
df_lda = pd.read_pickle(f"{DFDIR}/df_lda_final.pkl")
corpus, final_stopwords = prep['corpus'], prep['final_stopwords']

# ---------- Máscara de vacías, idéntica a la de 09_analisis_derivados ----------
cv  = CountVectorizer(stop_words=final_stopwords, min_df=10, max_df=0.6, ngram_range=(1, 2))
X   = cv.fit_transform(corpus)
vac = np.asarray(X.getnnz(axis=1) == 0)

corto = base['text'].astype(str).str.strip().str.len() < 15
b = base.assign(vacia=base['id'].isin(set(df_lda.loc[vac, 'id'])), corta=corto)

# ---------- Chequeos contra los valores de la tesis ----------
chk = {
    "n base == 27320":           len(base) == 27320,
    "matriz == (27102, 1857)":   X.shape == (27102, 1857),
    "vacías == 7404":            int(vac.sum()) == 7404,
    "cortas == 5349":            int(corto.sum()) == 5349,
    "df_lda alineado a corpus":  len(df_lda) == len(corpus),
}
for k, v in chk.items():
    print(f"  {'OK ' if v else 'FALLA'}  {k}")
assert all(chk.values()), "Algún chequeo falló: NO usar los resultados."

# ---------- Resultados con 2 decimales ----------
n   = len(base)
pos = base['stars'].isin([4, 5]).sum()
neg = base['stars'].isin([1, 2]).sum()
print(f"\nPolaridad  | n={n:,} | positivas (4-5)={pos:,} ({pos/n:.2%}) | negativas (1-2)={neg:,} ({neg/n:.2%})")
print(f"P(vacía | corta) = {b.loc[b.corta,'vacia'].mean():.2%}  "
      f"({b.loc[b.corta,'vacia'].sum():,} de {b.corta.sum():,})")
print(f"P(corta | vacía) = {b.loc[b.vacia,'corta'].mean():.2%}")

### Documentos cortos en el corpus de referencia (Sección 4.2.2)

In [ ]:
import pickle, numpy as np
DFDIR = "../Topic/Dataframes"
for ck, lab in [("ckpt_lda_multiseed.pkl","LDA"), ("ckpt_tfidf_multiseed.pkl","TF-IDF"),
                ("ckpt_w2v_multiseed.pkl","W2V"), ("ckpt_bertopic_multiseed.pkl","BERTopic")]:
    L = np.array([len(t) for t in pickle.load(open(f"{DFDIR}/{ck}", "rb"))['ref_texts']])
    print(f"{lab:<9} n={len(L):,}  <=1 token: {(L<=1).mean():.2%}  <=10 tokens: {(L<=10).mean():.2%}")